In [1]:
using Pkg; Pkg.activate(@__DIR__)

  Activating project at `/global/u1/b/blaschke/NetworkInterfaceControllers.jl/test`


In [2]:
Pkg.status()

Status `/global/u1/b/blaschke/NetworkInterfaceControllers.jl/test/Project.toml`
[1520ce14] AbstractTrees v0.4.5
  [0e44f5e4] Hwloc v3.3.0
  [0f8b85d8] JSON3 v1.14.3
  [6f74fd91] NetworkInterfaceControllers v1.0.0 `..`
  [6462fe0b] Sockets v1.11.0


In [3]:
using NetworkInterfaceControllers, Hwloc, AbstractTrees, JSON3

[ Info: Precompiling HwlocSelector [f69c1b03-cb70-55bc-8662-7ba15593e761] (cache misses: wrong dep version loaded (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [4]:
HwlocSelector = NetworkInterfaceControllers.get_hwloc_selector()

HwlocSelector

In [5]:
Hostlist = NetworkInterfaceControllers.Hostlists.HOSTLIST_TYPE[]

NetworkInterfaceControllers.Hostlists.SlurmHostlists.Hostlist

In [6]:
slurm_data = read(joinpath(@__DIR__, "perlmutter_slurm.json"), String) |> JSON3.read
slurm_feature_map = Dict{Symbol, Hostlist}()
for entry in slurm_data
    slurm_feature_map[Symbol(entry.feature)] = Hostlist(entry.hostlist)
end
slurm_feature_map

Dict{Symbol, NetworkInterfaceControllers.Hostlists.SlurmHostlists.Hostlist} with 10 entries:
  :cpu    => nid[200000-200255,004072-007143]
  :hbm40g => nid[001000-001001,001004-001005,001008-001009,001012-001013,001016…
  :cron   => login[01-40]
  :hbm80g => nid[200256-200257,200260-200261,200264-200265,200268-200269,200272…
  :milan  => nid[200000-200255,004072-007143]
  :gpu    => nid[200256-200257,200260-200261,200264-200265,200268-200269,200272…
  :ss11   => nid[200000-200257,200260-200261,200264-200265,200268-200269,200272…
  :a100   => nid[200256-200257,200260-200261,200264-200265,200268-200269,200272…
  :resv   => nid[200128-200255,200384-200385,200388-200389,200392-200393,200396…
  :pod    => nid[200000-200257,200260-200261,200264-200265,200268-200269,200272…

In [7]:
slurm_feature_map[:cpu] |> length

3328

In [8]:
slurm_feature_map[:a100] |> length

1920

In [9]:
slurm_feature_map[:gpu] |> length

1920

In [10]:
slurm_feature_map[:hbm40g] |> length

1536

In [11]:
slurm_feature_map[:hbm80g] |> length

384

In [12]:
(slurm_feature_map[:hbm40g] |> length) + (slurm_feature_map[:hbm80g] |> length)

1920

In [13]:
slurm_feature_map[:cpu]

nid[200000-200255,004072-007143]

In [14]:
cpu_list = slurm_feature_map[:cpu] |> string |> Hostlist |> Set
length(cpu_list)

3328

In [15]:
cpu_list_simple = Hostlist("nid[200000-200255,004072-007200]") |> Set
length(cpu_list_simple)

3385

In [16]:
intersect(cpu_list, cpu_list_simple) |> length

3328

In [17]:
slurm_feature_map[:gpu] 

nid[200256-200257,200260-200261,200264-200265,200268-200269,200272-200273,200276-200277,200280-200281,200284-200285,200288-200289,200292-200293,200296-200297,200300-200301,200304-200305,200308-200309,200312-200313,200316-200317,200320-200321,200324-200325,200328-200329,200332-200333,200336-200337,200340-200341,200344-200345,200348-200349,200352-200353,200356-200357,200360-200361,200364-200365,200368-200369,200372-200373,200376-200377,200380-200381,200384-200385,200388-200389,200392-200393,200396-200397,200400-200401,200404-200405,200408-200409,200412-200413,200416-200417,200420-200421,200424-200425,200428-200429,200432-200433,200436-200437,200440-200441,200444-200445,200448-200449,200452-200453,200456-200457,200460-200461,200464-200465,200468-200469,200472-200473,200476-200477,200480-200481,200484-200485,200488-200489,200492-200493,200496-200497,200500-200501,200504-200505,200508-200509,001000-001001,001004-001005,001008-001009,001012-001013,001016-001017,001020-001021,001024-001025,00

In [18]:
gpu_list = slurm_feature_map[:gpu] |> string |> Hostlist |> Set
length(gpu_list)

1920

In [19]:
gpu_list_simple = Hostlist("nid[200256-200512,000000-004071,008000-008800]") |> Set
length(gpu_list_simple)

5130

In [20]:
intersect(gpu_list, gpu_list_simple) |> length

1920

In [21]:
intersect(cpu_list, gpu_list)

Set{String}()

In [22]:
intersect(cpu_list_simple, gpu_list_simple)

Set{String}()

In [23]:
gethostname() in Hostlist("login[01-40],nid[200000-200255,004072-007200]")

true

In [24]:
import AbstractTrees: PreOrderDFS
import Hwloc: hwloc_pci_class_string

In [25]:
sys_devs = children(gettopology())
pci_devs = PreOrderDFS(sys_devs) |> collect |> filter(x->x.type==:PCI_Device)
net_devs = pci_devs |> filter(x->hwloc_pci_class_string(nodevalue(x).attr.class_id) == "Ethernet")

for dev in net_devs
    for io in dev.io_children
        name = io.object.name
        kind = io.object.subtype
        kind = kind == "" ? "Unknown" : kind
        println("Device $(name) is a $(kind) device")
    end
end

Device nmn0 is a Unknown device
Device net5 is a Unknown device
Device net2 is a Unknown device
Device mlx5_bond_0 is a Unknown device
Device net3 is a Unknown device
Device nmn1 is a Unknown device
Device net1 is a Unknown device
Device hsn0 is a Slingshot device
Device hsn1 is a Slingshot device


In [26]:
using Sockets

In [27]:
interfaces = get_interface_data(IPv4)

6-element Vector{Interface}:
 Interface("hsn0", :v4, ip"10.249.0.204")
 Interface("hsn1", :v4, ip"10.249.0.172")
 Interface("nmnb0", :v4, ip"10.252.1.113")
 Interface("hsnb0:chn", :v4, ip"128.55.64.19")
 Interface("hsnb0", :v4, ip"128.55.126.9")
 Interface("bond0", :v4, ip"128.55.167.25")

In [28]:
get_interface_data(IPv6)

6-element Vector{Interface}:
 Interface("hsn0", :v6, ip"fe80::ff:fe00:1f")
 Interface("hsn1", :v6, ip"fe80::ff:fe00:49")
 Interface("nmnb0", :v6, ip"fe80::1602:ecff:fee5:ee40")
 Interface("hsnb0", :v6, ip"2620:0:28b0:64::8037:4013")
 Interface("hsnb0", :v6, ip"fe80::1000:80ff:fe37:4013")
 Interface("bond0", :v6, ip"fe80::8ae9:a4ff:fe25:4572")

In [29]:
hsn0_public = filter(
    x->(x.name=="hsnb0:chn" && x.version==:v4), interfaces
) |> only 

Interface("hsnb0:chn", :v4, ip"128.55.64.19")

In [30]:
keys(ENV) |> collect .|> lowercase |> filter(x->startswith(x, "slurm"))

String[]

In [31]:
NICPreferences.NAME_SELECTOR

NetworkInterfaceControllers.NICPreferences.ModeSettings(NetworkInterfaceControllers.NICPreferences.USE_ALWAYS, nothing)

In [ ]:
NICPre